Build `rhythm_only_dataset.csv`

Extracts rhythm-only columns from model_dataset.csv and computes
four new annotation-derived features:
  - pct_ventricular           (from beat-level annotation files)
  - pct_supraventricular      (from beat-level annotation files)
  - rhythm_purity             (fraction of beats matching dominant rhythm)
  - episode_position_in_surgery (first episode start / case duration)

Saves result to NIBP_data/rhythm_only_dataset.csv

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

BASE_DIR = Path("..").resolve()
DATA_FINAL = BASE_DIR / "data" / "final"
ANN_DIR = BASE_DIR / "external_data" / "vitaldb-arrhythmia-database-1.0.0" / "Annotation_Files"

In [ ]:
# STEP 1: Extract rhythm-only base columns from model_dataset.csv
model_df = pd.read_csv(DATA_FINAL / 'model_dataset.csv')
print(f"model_dataset.csv loaded: {model_df.shape}")

ID_COLS     = [
    'caseid', 'episode_number', 'episode_start_sec', 'episode_end_sec',
    'bp_source', 'nibp_outcome_imputed', 'hypotension_label',
]
TEMP_COLS   = ['casestart', 'caseend']   # needed for computation, then dropped
RHYTHM_COLS = [
    'episode_duration_sec', 'episode_beat_count', 'episode_dominant_rhythm',
    'episode_beat_type', 'episode_rr_cv', 'dominant_rhythm',
]

keep = [c for c in (ID_COLS + TEMP_COLS + RHYTHM_COLS) if c in model_df.columns]
df   = model_df[keep].copy()
print(f"After column selection: {df.shape}  ({len(keep)} columns kept)")

In [ ]:
# ── STEP 2a: episode_position_in_surgery (model_dataset columns only) ─────────
# For each patient: first episode start / total case duration.
# casestart and caseend are dropped afterwards — they are clinical, not rhythm.

case_stats = (
    df.groupby('caseid')
    .agg(
        first_ep_start=('episode_start_sec', 'min'),
        casestart_val=('casestart', 'first'),
        caseend_val=('caseend', 'first'),
    )
    .reset_index()
)
case_stats['case_duration'] = case_stats['caseend_val'] - case_stats['casestart_val']

bad_dur = case_stats[case_stats['case_duration'] <= 0]
for _, row in bad_dur.iterrows():
    print(f"WARNING: caseid={int(row['caseid'])} has case_duration={row['case_duration']} "
          f"— setting episode_position_in_surgery to NaN")

case_stats['episode_position_in_surgery'] = np.where(
    case_stats['case_duration'] > 0,
    case_stats['first_ep_start'] / case_stats['case_duration'],
    np.nan,
)

df = df.merge(case_stats[['caseid', 'episode_position_in_surgery']], on='caseid', how='left')
df = df.drop(columns=['casestart', 'caseend'])
print(f"episode_position_in_surgery computed. Shape now: {df.shape}")


In [ ]:
# ── STEP 2b: Annotation-derived features ─────────────────────────────────────
# Loads each patient's annotation file once and computes:
#   pct_ventricular      — V beats / all non-normal beats in episode window
#   pct_supraventricular — S beats / all non-normal beats in episode window
#   rhythm_purity        — beats matching episode_dominant_rhythm / total beats

caseids     = df['caseid'].unique()
n_cases     = len(caseids)
case_groups = df.groupby('caseid')

print(f"Computing annotation features for {len(df)} episodes across {n_cases} patients.\n")

missing_ann = []
records     = []

for i, caseid in enumerate(caseids, 1):
    if i == 1 or i % 50 == 0 or i == n_cases:
        print(f"  [{i}/{n_cases}] caseid={caseid}")

    ann_path = os.path.join(ANN_DIR, f"Annotation_file_{caseid}.csv")
    group    = case_groups.get_group(caseid)

    ann = None
    if not os.path.exists(ann_path):
        missing_ann.append(caseid)
    else:
        try:
            ann = pd.read_csv(ann_path)
            # Quality filter: remove beats flagged as bad signal
            if 'bad_signal_quality' in ann.columns:
                ann = ann[ann['bad_signal_quality'] != True].copy()
        except Exception:
            missing_ann.append(caseid)
            ann = None

    for _, row in group.iterrows():
        if ann is None:
            records.append({
                'caseid': row['caseid'], 'episode_number': row['episode_number'],
                'pct_ventricular': np.nan, 'pct_supraventricular': np.nan,
                'rhythm_purity': np.nan,
            })
            continue

        s_sec      = row['episode_start_sec']
        e_sec      = row['episode_end_sec']
        dom_rhythm = row['episode_dominant_rhythm']

        # Filter to beats within the episode duration window
        ep      = ann[(ann['time_second'] >= s_sec) & (ann['time_second'] <= e_sec)]
        n_total = len(ep)

        # pct_ventricular / pct_supraventricular
        if 'beat_type' in ep.columns and n_total > 0:
            non_normal = ep[ep['beat_type'].notna() & (ep['beat_type'] != 'N')]
            n_nn = len(non_normal)
            if n_nn > 0:
                pct_v = float((non_normal['beat_type'] == 'V').sum() / n_nn)
                pct_s = float((non_normal['beat_type'] == 'S').sum() / n_nn)
            else:
                pct_v = np.nan
                pct_s = np.nan
        else:
            pct_v = np.nan
            pct_s = np.nan

        # rhythm_purity
        if 'rhythm_label' in ep.columns and n_total > 0 and pd.notna(dom_rhythm):
            purity = float((ep['rhythm_label'] == dom_rhythm).sum() / n_total)
        else:
            purity = np.nan

        records.append({
            'caseid':               row['caseid'],
            'episode_number':       row['episode_number'],
            'pct_ventricular':      pct_v,
            'pct_supraventricular': pct_s,
            'rhythm_purity':        purity,
        })

if missing_ann:
    print(f"\nAnnotation files missing for {len(missing_ann)} patient(s): "
          f"{missing_ann[:10]}{'...' if len(missing_ann) > 10 else ''}")

feat_df   = pd.DataFrame(records)
rhythm_df = df.merge(feat_df, on=['caseid', 'episode_number'], how='left')
print(f"\nFinal shape: {rhythm_df.shape}")


In [ ]:
# ── VERIFICATION CHECKS ───────────────────────────────────────────────────────
NEW_FEATS = [
    'pct_ventricular', 'pct_supraventricular',
    'rhythm_purity', 'episode_position_in_surgery',
]
FORBIDDEN_PATTERNS = [
    'map_dur', 'hr_dur', 'spo2_dur',
    'preop_', 'lab_', 'intraop_',
    'casestart', 'caseend',
]

# CHECK 1: Shape and forbidden columns
print("=" * 70)
print("CHECK 1: Shape and forbidden column check")
print("=" * 70)
print(f"  Rows: {len(rhythm_df)}  (expected 1,284)")
print(f"  Cols: {len(rhythm_df.columns)}")
print(f"  Columns: {list(rhythm_df.columns)}")
forbidden_found = [
    col for col in rhythm_df.columns
    if any(pat in col for pat in FORBIDDEN_PATTERNS)
]
if forbidden_found:
    print(f"  WARNING: Forbidden columns found — removing: {forbidden_found}")
    rhythm_df = rhythm_df.drop(columns=forbidden_found)
else:
    print("  OK: No forbidden columns present.")
assert len(rhythm_df) == 1284, f"FAIL: {len(rhythm_df)} rows (expected 1284)"

# CHECK 2: Alignment with model_dataset (5 random episodes)
print("\n" + "=" * 70)
print("CHECK 2: Rhythm column alignment with model_dataset (5 random episodes)")
print("=" * 70)
check_cols = [
    'caseid', 'episode_number', 'episode_dominant_rhythm',
    'episode_rr_cv', 'episode_duration_sec', 'episode_beat_count',
]
rng        = np.random.default_rng(seed=789)
sample_idx = rng.choice(len(rhythm_df), size=5, replace=False)
all_match  = True
for idx in sample_idx:
    r_row = rhythm_df.iloc[idx]
    cid, ep = r_row['caseid'], r_row['episode_number']
    m_row = model_df[(model_df['caseid'] == cid) & (model_df['episode_number'] == ep)].iloc[0]
    mismatches = []
    for col in check_cols:
        if col not in rhythm_df.columns or col not in model_df.columns:
            continue
        rv, mv = r_row[col], m_row[col]
        if not (pd.isna(rv) and pd.isna(mv)) and rv != mv:
            mismatches.append(f"{col}: rhythm_only={rv} vs model={mv}")
    status = "MATCH" if not mismatches else f"MISMATCH ({'; '.join(mismatches)})"
    if mismatches:
        all_match = False
    print(f"  caseid={int(cid)}, episode={int(ep)}: {status}")
print(f"  {'ALL MATCH — OK' if all_match else 'SOME MISMATCHES — investigate'}")

# CHECK 3: Value ranges (must be 0–1)
print("\n" + "=" * 70)
print("CHECK 3: New feature value ranges (must be strictly 0–1)")
print("=" * 70)
for feat in NEW_FEATS:
    vals = rhythm_df[feat].dropna()
    if len(vals) == 0:
        print(f"  {feat}: ALL NaN — WARNING")
        continue
    mn, mx, mean = vals.min(), vals.max(), vals.mean()
    flag = "  WARNING: values outside [0, 1]" if (mn < 0 or mx > 1) else ""
    print(f"  {feat}: mean={mean:.4f}  min={mn:.4f}  max={mx:.4f}{flag}")

# CHECK 4: pct_ventricular + pct_supraventricular <= 1.0
print("\n" + "=" * 70)
print("CHECK 4: pct_ventricular + pct_supraventricular <= 1.0")
print("=" * 70)
both_valid = rhythm_df[['pct_ventricular', 'pct_supraventricular']].dropna()
violations = int(
    ((both_valid['pct_ventricular'] + both_valid['pct_supraventricular']) > 1.0 + 1e-9).sum()
)
print(f"  Rows with both non-null: {len(both_valid)}")
print(f"  Violations (sum > 1.0):  {violations}  {'OK' if violations == 0 else 'FAIL'}")

# CHECK 5: Missingness
print("\n" + "=" * 70)
print("CHECK 5: Missingness for new features")
print("=" * 70)
n = len(rhythm_df)
for feat in NEW_FEATS:
    pct  = rhythm_df[feat].isna().sum() / n * 100
    flag = ""
    if feat in ('pct_ventricular', 'pct_supraventricular') and pct > 20:
        flag = "  WARNING: >20% missing"
    if feat == 'episode_position_in_surgery' and pct > 1:
        flag = (f"  NOTE: {pct:.1f}% missing — caused by patients whose casestart/"
                "caseend were NaN during the NaN-episode restoration step")
    print(f"  {feat}: {pct:.1f}% missing{flag}")


In [ ]:
# SAVE
out_path = DATA_FINAL / 'rhythm_only_dataset.csv'
rhythm_df.to_csv(out_path, index=False)
print(f"Saved: rhythm_only_dataset.csv  {rhythm_df.shape}")